# AIEA Task 7: Backwards Chaining
In this notebook, I will implement a backwards chaining system for First Order Logic (FOL) using the guidelines from the CSE240 Assignment 4 repo @ https://github.com/ucsc-cse-240/assignments/blob/main/Assignment4/Assignment4.ipynb

## What is backwards chaining?
- Start with a specific goal
- Check database of known facts 
    - Check what you know
- Find rule with matching consequence 
    - Each condition becomes a sub-goal
- Recursively attempt to prove goal
    - Antecedents (conditions) are tried from left to right in the order they appear
- If no assertion matches and no rule's consequent matches your goal, the system assumes the goal is false. (closed world assumption)

### When do we use backwards chaining?
In a fan-out situation where you start with lots of facts and have a specific goal. When you don't have your facts yet and want to test a specific hypothesis, backward chaining is great because it effectively tells you which facts to go gather. Each sub-goal it generates is essentially a question: "go find out if this is true." It focuses your information-gathering effort.

## First, Forward Chaining
Write a one-rule system that finds all other combinations of which poker hands beat which, transitively, given some of the rankings already. For example, it should be able to deduce that a three-of-a-kind beats a pair, because a three-of-a-kind beats two-pair and a two-pair beats a pair. The rankings (data) are all provided in the form '(?x) beats (?y)'.

This demonstrates Forward Chaining by using the transitive rule to deduce new rules based off of our initial rule set.

In [11]:
# Imports for implementing chaining
from production import IF, AND, OR, NOT, THEN, DELETE, forward_chain
from data import *

In [12]:
def transitive_rule():
    rule = IF( AND( '(?x) beats (?y)',
                    '(?y) beats (?z)' ),
               THEN( '(?x) beats (?z)' ))
    return rule

In [13]:
# Test rule using forward chaining
print(forward_chain([transitive_rule()], abc_data))
print(forward_chain([transitive_rule()], poker_data))
print(forward_chain([transitive_rule()], minecraft_data))

('a beats b', 'b beats c', 'a beats c')
('two-pair beats pair', 'three-of-a-kind beats two-pair', 'straight beats three-of-a-kind', 'flush beats straight', 'full-house beats flush', 'straight-flush beats full-house', 'three-of-a-kind beats pair', 'straight beats two-pair', 'straight beats pair', 'flush beats three-of-a-kind', 'flush beats two-pair', 'flush beats pair', 'full-house beats straight', 'full-house beats three-of-a-kind', 'full-house beats two-pair', 'full-house beats pair', 'straight-flush beats flush', 'straight-flush beats straight', 'straight-flush beats three-of-a-kind', 'straight-flush beats two-pair', 'straight-flush beats pair')
('diamond-sword beats diamond-axe', 'stone-pick beats stone-shovel', 'diamond-axe beats iron-axe', 'iron-axe beats stone-shovel', 'iron-pick beats stone-pick', 'iron-axe beats iron-pick', 'stone-shovel beats fist', 'diamond-sword beats iron-axe', 'stone-pick beats fist', 'diamond-axe beats stone-shovel', 'diamond-sword beats stone-shovel', 'd

## Creating a rule set for family relations

This shows how to create a rule set that can be used to define relationships given some data. In this case we test the family relations rule set on the Simpsons data set and a custom data set to verify that our rules generate correct relations.

In [14]:
def family_rules():
    rules = [ IF( 'person (?x)',
                     THEN('self (?x) (?x)') ),

                 IF( AND('parent (?p) (?x)',
                         'parent (?p) (?y)',
                         NOT('self (?x) (?y)')),
                     THEN('sibling (?x) (?y)') ),
                 
                 IF( 'parent (?y) (?x)',
                     THEN('child (?x) (?y)')),
                 
                 IF( AND( 'parent (?x) (?y)',
                            'parent (?y) (?z)' ),
                     THEN( 'grandparent (?x) (?z)' )),
                 
                 IF( 'grandparent (?x) (?y)',
                     THEN( 'grandchild (?y) (?x)' )),
                 
                 IF( AND( 'parent (?a) (?x)',
                         'parent (?b) (?y)',
                         'sibling (?a) (?b)',
                         NOT( 'self (?x) (?y)' ),
                         NOT( 'sibling (?x) (?y)' )),
                     THEN( 'cousin (?x) (?y)' )),
            ]
    return rules


# Test data on Simpsons family
print(forward_chain(family_rules(), simpsons_data, verbose=False))

# The following should generate 14 cousin relationships, representing 7 pairs
# of people who are cousins:
black_family_cousins = [
     relation for relation in
     forward_chain(family_rules(), black_data, verbose=False)
     if "cousin" in relation ]

# See if all relations were found
print(black_family_cousins)

('person bart', 'person lisa', 'person maggie', 'person marge', 'person homer', 'person abe', 'parent marge bart', 'parent marge lisa', 'parent marge maggie', 'parent homer bart', 'parent homer lisa', 'parent homer maggie', 'parent abe homer', 'self bart bart', 'self lisa lisa', 'self maggie maggie', 'self marge marge', 'self homer homer', 'self abe abe', 'sibling bart lisa', 'sibling bart maggie', 'sibling lisa bart', 'sibling lisa maggie', 'sibling maggie bart', 'sibling maggie lisa', 'child bart marge', 'child lisa marge', 'child maggie marge', 'child bart homer', 'child lisa homer', 'child maggie homer', 'child homer abe', 'grandparent abe bart', 'grandparent abe lisa', 'grandparent abe maggie', 'grandchild bart abe', 'grandchild lisa abe', 'grandchild maggie abe')
['cousin sirius bellatrix', 'cousin sirius andromeda', 'cousin sirius narcissa', 'cousin regulus bellatrix', 'cousin regulus andromeda', 'cousin regulus narcissa', 'cousin bellatrix sirius', 'cousin bellatrix regulus', '

## Backwards Chaining
Now we will do backwards chaining starting from a conclusion and generating a goal tree of all goals we may need to test.

All variables that appear in a rule's antecedent also appear in its consequent (so there are no "unknown" variables in the antecedent). In other words, you will not need to do backtracking. All assertions are positive: no rules will have DELETE clauses or NOT expressions. Rule antecedents never have nested RuleExpression nodes. For example, an expression such as (OR (AND x y) (AND z w)) will never appear within an antecedent, because that contains an AND expression nested under an OR expression. Rule consequents always have just a single statement. Note that an antecedent can be a single hypothesis (a string) or a RuleExpression.

### Implementation
- Use a recursive structure that checks for matching rules for the input hypothesis
- If there are matching rules that bind, find the antecedent (or condition) that preceeds the consequence
- If the antecedent is a string and not an AND or OR, then we can simply just add it to the list of rules to verify
- If the antecednet is an AND or OR object, we must recursively backwards chain through each of the conditions, adding the populated antecedents to the result until we reach a string antecedent
- Once we find all rules, we can return the list of rules that we must verify to arrive at the desired conclusion

In [ ]:
# Import additional methods for backchaining
from production import AND, OR, NOT, PASS, FAIL, IF, THEN, match, populate, simplify, variables
from data import zookeeper_rules

def backchain_to_goal_tree(rules, hypothesis):
    """
    Takes a hypothesis (string) and a list of rules (list
    of IF objects), returning an AND/OR tree representing the
    backchain of possible statements we may need to test
    to determine if this hypothesis is reachable or not.

    This method should return an AND/OR tree, that is, an
    AND or OR object, whose constituents are the subgoals that
    need to be tested. The leaves of this tree should be strings
    (possibly with unbound variables), *not* AND or OR objects.
    Make sure to use simplify(...) to flatten trees where appropriate.
    """
    res = [hypothesis]  # Start tree with just the hypothesis
    
    for rule in rules:
        # Find rules that have a consequence that matches the hypothesis
        bindings = match(rule.consequent(), hypothesis)
        if bindings is not None:
            # Get the antecedent, or the condition
            antecedent = rule.antecedent()
            # Populate the antecedent with the variables in the rule
            populated = populate(antecedent, bindings)
            
            if isinstance(antecedent, str):
                # If the antecedent is a string instead of an AND or OR object, append to results
                res.append(backchain_to_goal_tree(rules, populated))
            if isinstance(antecedent, AND):
                subgoals = []
                for condition in populated:
                    subgoals.append(backchain_to_goal_tree(rules, condition))
                res.append(AND(*subgoals))
            if isinstance(antecedent, OR):
                subgoals = []
                for condition in populated:
                    subgoals.append(backchain_to_goal_tree(rules, condition))
                res.append(OR(*subgoals))
                
    # Don't forget to simplify the tree    
    return simplify(OR(*res))


# Uncomment this to test out your backward chainer:
print(backchain_to_goal_tree(zookeeper_rules, 'opus is a penguin'))

OR('opus is a penguin', AND(OR('opus is a bird', 'opus has feathers', AND('opus flies', 'opus lays eggs')), 'opus does not fly', 'opus swims', 'opus has black and white color'))


## Conclusion

We can see that by using backwards chaining, we were able to obtain the correct set of rules/goals that we would need to test in order to verify our hypothesis/conclusion. In this lab, I was able to successfully implement a backward chaining inference algorithm for First Order Logic (FOL). 